# IOAI — 2025 Team Selection Day2 Ml Player Clustering (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day2-ml-player-clustering'
for f in ['train.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 2 — Player Clustering · 모범답안 (Reference Solution)

> **Kazakhstan – Team Selection 2025 · Day 2**

Faithful reproduction of the strong approach (Batyr Yerdenov's TST solution): **separate goalkeepers** from outfield players (their gk_* attributes dominate), **standardise each group**, choose the number of clusters by **silhouette search**, fit a **Gaussian Mixture Model** per group, and merge with **disjoint cluster ids**. Writes `submission.csv` (`id,cluster`) for Kaggle B-Cubed F1 grading.

In [ ]:
import os, pandas as pd, numpy as np
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day2_ML_Player_Clustering"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."
DATA=_find("train.csv")
df=pd.read_csv(os.path.join(DATA,"train.csv"))   # id + 34 FIFA-style attributes (incl gk_*)
feat=[c for c in df.columns if c!="id"]
print("players",df.shape,"| features",len(feat))


## Split goalkeepers vs outfield

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
gk_cols=[c for c in feat if c.startswith("gk_")]
out_cols=[c for c in feat if not c.startswith("gk_")]
imp=SimpleImputer(strategy="mean")
F=pd.DataFrame(imp.fit_transform(df[feat]), columns=feat)
is_gk = F[gk_cols].mean(1) > 40          # goalkeepers have high gk_* ratings
print("goalkeepers:", int(is_gk.sum()), "/ outfield:", int((~is_gk).sum()))


## GMM per group with silhouette-chosen k

In [ ]:
def cluster_group(Fsub, cols, k_range):
    X=StandardScaler().fit_transform(Fsub[cols])
    best=None
    for k in k_range:
        lab=GaussianMixture(n_components=k, covariance_type="diag", random_state=0).fit_predict(X)
        s=silhouette_score(X[:4000], lab[:4000]) if len(X)>k else -1
        if best is None or s>best[0]: best=(s,k,lab)
    print(f"  chosen k={best[1]} (silhouette {best[0]:.4f})")
    return best[2]
lab_out = cluster_group(F[~is_gk], out_cols, range(5,13))
lab_gk  = cluster_group(F[is_gk],  gk_cols,  range(2,5))


## Merge with disjoint ids → `submission.csv`

In [ ]:
clusters=np.zeros(len(df), dtype=int)
clusters[(~is_gk).values]=lab_out
clusters[(is_gk).values]=lab_gk + (lab_out.max()+1)   # offset so gk ids never collide with outfield ids
pd.DataFrame({"id":df["id"], "cluster":clusters}).to_csv("submission.csv", index=False)
print("total clusters:", len(np.unique(clusters)), "| wrote submission.csv")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)